# Reconnect / re-remap ToF sensors

Mid-session recovery when the sensors fall off the bus (bad contact, XSHUT wire
unplugged, power dip). No reboot needed — the remap address lives in a volatile
register, so any power loss or XSHUT-low resets a sensor to `0x29`.

**TLDR: run the next cell and click `Resolve!`** — it auto-detects the case and
applies the right fix. The cells below it are the same steps run manually, for
when you want to see each stage.

Reminder: **0x30 = always-on sensor** (renamed), **0x29 = sensor on XSHUT pin 29**
(only visible while pin 29 is held HIGH).

| i2cdetect shows | meaning | fix |
|---|---|---|
| `29` + `30` | healthy | nothing to do |
| only `30` | XSHUT sensor asleep | Fix A cell |
| only `29` | both reset, colliding on 0x29 | Fix B cell |
| neither | power/wiring dead | reseat VIN / check cables / reboot |

In [ ]:
# TLDR — one click: auto-detect the case and apply the right fix.
# (remap_utils.py must sit in the same folder as this notebook)
from remap_utils import remap_button
remap_button()

In [ ]:
# 1. DETECT — scans bus 1 and prints which case you are in.
import re, subprocess

p = subprocess.run(
    ["sudo", "-S", "i2cdetect", "-y", "-r", "1"],
    input="jetson\n", capture_output=True, text=True,
)
if p.returncode != 0:
    print(p.stderr)
    raise SystemExit("i2cdetect failed — is /dev/i2c-1 there?")
print(p.stdout)

found = set()
for line in p.stdout.splitlines():
    if re.match(r"^[0-7]0:", line):        # grid rows look like '30: -- -- 33 ...'
        found.update(line[4:].split())     # skip the row label, keep cell values

has29, has30 = "29" in found, "30" in found
if has29 and has30:
    print("OK: both sensors present (0x29 + 0x30). Nothing to do.")
elif has30:
    print("CASE A: only 0x30 — XSHUT sensor is asleep (pin 29 not held HIGH). Run the Fix A cell below.")
elif has29:
    print("CASE B: only 0x29 — both sensors reset and collide on 0x29. Run the Fix B cell below.")
else:
    print("HARDWARE: neither address on the bus. Check sensor power/wiring, reseat VIN, or reboot.")

In [ ]:
# 2. FIX A — only 0x30 visible: the XSHUT sensor just needs pin 29 HIGH to wake at 0x29.
#    (echo | sudo -S pipes the password so Jupyter never prompts)
!echo 'jetson' | sudo -S systemctl stop tof-i2c-switcher-simple.service  # free the pin (else gpioset: busy)
!echo 'jetson' | sudo -S gpioset --mode=exit $(gpiofind PQ.05)=0         # pin 29 LOW: XSHUT sensor deaf
!echo 'jetson' | sudo -S gpioset --mode=exit $(gpiofind PQ.05)=1         # pin 29 HIGH: XSHUT sensor wakes at 0x29
# re-hold the pin via the service so it stays HIGH after this cell
# (restart = stop releases pin, start re-holds; its rename is a harmless no-op here)
# !echo 'jetson' | sudo -S systemctl restart tof-i2c-switcher-simple.service
!echo 'jetson' | sudo -S i2cdetect -y -r 1   # expect: 29 + 30

In [ ]:
# 3. FIX B — only 0x29 visible: both sensors reset and collide on 0x29.
#    Silence the XSHUT sensor, rename the always-on one, wake XSHUT again.
!echo 'jetson' | sudo -S systemctl stop tof-i2c-switcher-simple.service  # free the pin (else gpioset: busy)
!echo 'jetson' | sudo -S busybox devmem 0x2430068 w 0x8                  # pinmux (set at boot; harmless to redo)
!echo 'jetson' | sudo -S gpioset --mode=exit $(gpiofind PQ.05)=0         # pin 29 LOW: XSHUT sensor deaf
!echo 'jetson' | sudo -S i2ctransfer -y 1 w2@0x29 0x8A 0x30              # rename the other one: 0x29 -> 0x30
!echo 'jetson' | sudo -S gpioset --mode=exit $(gpiofind PQ.05)=1         # pin 29 HIGH: XSHUT sensor wakes at 0x29
# !echo 'jetson' | sudo -S systemctl start tof-i2c-switcher-simple.service # service holds pin 29 HIGH from now on
!echo 'jetson' | sudo -S i2cdetect -y -r 1                               # expect: 29 + 30

**Notes**

- **Neither address** → no software fix. Reseat the sensor connectors / VIN, then
  reboot (the boot service redoes the whole remap automatically).
- `Device or resource busy` from a `gpioset` line means the service is already holding
  pin 29 — run the `systemctl stop` line of Fix B first, or just click **Resolve!**.
- These fixes leave the service **stopped** (the `start`/`restart` lines are commented
  out). The pin stays HIGH on this hardware after `gpioset` exits, but nothing is
  holding it — `sudo systemctl start tof-i2c-switcher-simple` or a reboot restores
  the auto-hold.
- The sudo password lives in `remap_utils.py` (`SUDO_PASSWORD`) and in the manual
  cells. If you changed the default `jetson` password, update both. Plaintext —
  don't share this notebook or the util as-is.
- The `[sudo] password` line in the output is just prompt text, not an error.
- Afterwards, re-run `test_two_sensors.py` or your JetRacer notebook to confirm ranging.